## Task 3 - Chatbot using Hugging Face Transformers 

In [4]:
# Install required libraries (run once)

%pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Import required libraries

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

c:\Users\DELL\OneDrive\Desktop\ALL INTERNSHIP WORK\GEN_AI_MODULE\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Check if GPU available

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


In [7]:
# Load pre-trained chatbot model (DialoGPT-small)

model_name = "microsoft/DialoGPT-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.to(device)

c:\Users\DELL\OneDrive\Desktop\ALL INTERNSHIP WORK\GEN_AI_MODULE\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--microsoft--DialoGPT-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 149/149 [00:00<00:00, 3167.55it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [8]:
# Check if model loaded properly

print("Model and tokenizer loaded successfully!")

Model and tokenizer loaded successfully!


In [15]:
def generate_response(user_input, chat_history_ids=None):
    
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt').to(device)
    
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids
    
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long).to(device)
    
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        
        do_sample=True,              # enables randomness
        temperature=0.7,             # controls creativity
        top_k=50,                    # limits choices
        top_p=0.9,                   # nucleus sampling
        
        no_repeat_ngram_size=3       # prevents repetition
    )
    
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )
    
    return response, chat_history_ids

In [16]:
chat_history = None

user_input = "Hi"
response, chat_history = generate_response(user_input, chat_history)

print("User:", user_input)
print("Bot :", response)

User: Hi
Bot : Hi


Chatbot: Hello! I am your AI assistant. How can I help you today?

User: Hello  
Chatbot: Hello! Nice to meet you. How can I assist you?

User: What is Artificial Intelligence?  
Chatbot: Artificial Intelligence is the simulation of human intelligence in machines.

User: Who created Python?  
Chatbot: Python was created by Guido van Rossum.

User: exit  
Chatbot: Goodbye!

In [17]:
# Interactive chatbot

print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.\n")

chat_history = None

while True:
    user_input = input("User: ")
    
    # Exit condition
    if user_input.lower() in ['exit', 'quit']:
        print("Chatbot: Goodbye!")
        break
    
    # Generate response
    response, chat_history = generate_response(user_input, chat_history)
    
    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.

Chatbot: Hi!
Chatbot: It's not that complicated. It's just that the AI can't do it.
Chatbot: Python is not a language.
Chatbot: This is why I love dogs.
Chatbot: Goodbye!


### Conclusion
A chatbot was successfully built using a pre-trained transformer model from Hugging Face.
The DialoGPT-small model was used for generating conversational responses.
*The chatbot can:*
- Accept user input
- Generate meaningful responses
- Maintain conversation context
- Exit on command

<br>

*Techniques used:*   
- Tokenization using Hugging Face tokenizer
- Text generation using transformer model
- Attention mask for stable responses
- Sampling techniques (top-k, top-p, temperature) for better output  
<br>

*Limitations:*
- Responses may not always be accurate
- Model performance is limited due to lightweight architecture  

<br>

Overall, this project demonstrates how transformer-based models can be used to build interactive conversational AI systems.